# 02 - Dataset

In [12]:
import json
from pathlib import Path
import numpy as np

root = Path("../data/processed/shapes_v1")
manifest = json.loads((root / "manifest.json").read_text())
splits = ["train", "val", "test_iid", "test_comp"]
data = {s: np.load(root / f"{s}.npz") for s in splits}

print(manifest["counts"])
d = data["train"]
for k in d.files:
    print(f"{k:10s} {d[k].shape}  {d[k].dtype}")
print(d["captions"][:3])

{'train': 200000, 'val': 5000, 'test_iid': 5000, 'test_comp': 10000}
images     (200000, 16, 16)  uint8
n_obj      (200000,)  uint8
color      (200000, 2)  int8
shape      (200000, 2)  int8
size       (200000, 2)  int8
pos        (200000,)  int8
rel        (200000,)  int8
jitter     (200000, 2, 2)  int8
captions   (200000,)  <U56
['a large cyan triangle above a small white square'
 'a small orange cross right of a small orange square'
 'a large white cross above a small cyan circle']


In [11]:
import numpy as np
aux = np.array(
    [
        [i for i in range(0,256)] 
        for _ in range(0, 151)
    ]
)

aux1 = aux.reshape(1, -1)
aux, aux1

(array([[  0,   1,   2, ..., 253, 254, 255],
        [  0,   1,   2, ..., 253, 254, 255],
        [  0,   1,   2, ..., 253, 254, 255],
        ...,
        [  0,   1,   2, ..., 253, 254, 255],
        [  0,   1,   2, ..., 253, 254, 255],
        [  0,   1,   2, ..., 253, 254, 255]], shape=(151, 256)),
 array([[  0,   1,   2, ..., 253, 254, 255]], shape=(1, 38656)))

In [19]:
npz_dict = dict(np.load("../data/processed/shapes_v1/train.npz"))
npz_dict

{'images': array([[[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         ...,
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         ...,
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         ...,
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        ...,
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         ...,
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ...,

In [20]:
npz_dict.keys()

dict_keys(['images', 'n_obj', 'color', 'shape', 'size', 'pos', 'rel', 'jitter', 'captions'])

In [21]:
npz_dict["captions"]

array(['a large cyan triangle above a small white square',
       'a small orange cross right of a small orange square',
       'a large white cross above a small cyan circle', ...,
       'a small green triangle below a small blue triangle',
       'a large yellow cross left of a small orange cross',
       'a small cyan triangle left of a large purple triangle'],
      shape=(200000,), dtype='<U56')

In [22]:
npz_dict["color"]


array([[7, 6],
       [5, 5],
       [6, 7],
       ...,
       [2, 1],
       [3, 5],
       [7, 4]], shape=(200000, 2), dtype=int8)

In [26]:
manifest.keys()

dict_keys(['name', 'seed', 'config', 'counts', 'sha256', 'text_vocab', 'palette_rgb', 'numpy_version'])

In [32]:
import numpy as np
import torch
from torch.utils.data import Dataset

class ShapesDataset(Dataset):
    def __init__(self, npz_path, tokenizer, max_len, n_colors):
        # 1) Carga los arrays UNA vez en memoria: dict(np.load(npz_path))
        arrays = dict(np.load(npz_path))
        # 2) Guarda captions, images y los atributos que necesites
        self.images = arrays["images"].reshape(len(arrays["images"]), -1)
        self.captions = arrays["captions"]
        self.src = np.array(
            [tokenizer.encode(c, max_len) for c in self.captions],
            dtype=np.int64
        )
        self.bos_id = n_colors
        self.pad_id = tokenizer.pad_id

    def __len__(self):
        return len(self.src)

    def __getitem__(self, i):
        # devuelve un diccionario de tensores
        pixels = torch.from_numpy(self.images[i]).long() # (256, )
        bos = torch.tensor([self.bos_id])
        tgt_in = torch.cat([bos, pixels[:-1]])
        src = torch.from_numpy(self.src[i])
        src_pad_mask = src == self.pad_id

        return {
            "src": src,
            "src_pad_mask": src_pad_mask,
            "tgt_in": tgt_in,
            "tgt_out": pixels,
        }

In [28]:
c = d["color"]
c = c.ravel()
c = c[ c >= 0 ]
np.bincount(c)

array([42731, 42985, 42639, 42332, 56398, 56877, 56761, 57261])

In [34]:
tok = TextTokenizer(manifest["text_vocab"])
max_len = max(len(c.split()) for s in splits for c in data[s]["captions"])   # 10
n_colors = len(manifest["palette_rgb"])                                       # 9

sd = ShapesDataset(root / "train.npz", tok, max_len, n_colors)
item = sd[0]
print({k: (v.shape, v.dtype) for k, v in item.items()})
print(item["tgt_in"][:5], item["tgt_out"][:5])

{'src': (torch.Size([10]), torch.int64), 'src_pad_mask': (torch.Size([10]), torch.bool), 'tgt_in': (torch.Size([256]), torch.int64), 'tgt_out': (torch.Size([256]), torch.int64)}
tensor([9, 0, 0, 0, 0]) tensor([0, 0, 0, 0, 0])
